# Stage 04 — Data Acquisition and Ingestion (Homework)

**Goals:** API pull, scraping, secrets via `.env`, validation, saving to `data/raw/`.

> Ethics & legality: obey site Terms, robots.txt, and rate limits. Do not scrape where prohibited.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
from pathlib import Path
from dotenv import load_dotenv

HOMEWORK_DIR = Path.cwd()
ROOT = HOMEWORK_DIR
REPO_ROOT = HOMEWORK_DIR.parents[1]

load_dotenv(HOMEWORK_DIR / ".env")

print("HOMEWORK_DIR:", HOMEWORK_DIR)
print("ROOT:", ROOT)
print("REPO_ROOT:", REPO_ROOT)

HOMEWORK_DIR: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/homework/homework4
ROOT: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/homework/homework4
REPO_ROOT: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong


In [3]:
from pathlib import Path
from dotenv import load_dotenv
import os

HOMEWORK_DIR = Path.cwd()
REPO_ROOT = HOMEWORK_DIR.parents[1]

load_dotenv(HOMEWORK_DIR / ".env")

print("Homework directory:", HOMEWORK_DIR)
print("Repository root:", REPO_ROOT)    # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Homework directory: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/homework/homework4
Repository root: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong
Looking in: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/homework/homework4

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [4]:
import os, json, time, datetime as dt, csv, pathlib
from typing import Dict, List
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

DATA_RAW = HOMEWORK_DIR / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")

# No key yet? A free one takes about 30 seconds:
#     https://www.alphavantage.co/support/#api-key
# Put the API key in the .env file at the project root:
#     ALPHAVANTAGE_API_KEY=your_key_here
# Never hard-code it in the notebook - notebooks get shared, committed and screen-shared.
# Without a key this notebook still runs; it falls back to yfinance below.
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))

Loaded ALPHAVANTAGE_API_KEY? True


## Helper functions: validation & filenames

In [5]:
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

In [6]:
def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join([f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()])
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

In [7]:
def validate_df(df: pd.DataFrame, required_cols: List[str], dtypes_map: Dict[str, str]) -> Dict[str, str]:
    msgs = {}
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"
    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = f"Failed to coerce {col} to {dtype}: {e}"
    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"
    return msgs

## API Ingestion: Alpha Vantage (fallback: yfinance)
- If `ALPHAVANTAGE_API_KEY` is set, use Alpha Vantage `TIME_SERIES_DAILY`.
- Else, demonstrate with `yfinance` (no key).
- **Adjusted close is a paid field at Alpha Vantage**, so both branches take the raw `close`. Worth pausing on: a vendor moved a column behind a paywall, and any pipeline that assumed the column existed broke. Code defensively against the *shape* of what comes back, not the shape you remember.
- The free tier allows **25 calls a day**. Past that the API still answers `200 OK` with an explanation instead of data - which is why the code checks for the series rather than trusting the status code.

In [8]:
SYMBOL = "AAPL"

use_alpha = bool(ALPHA_KEY)
print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": SYMBOL,
        "outputsize": "compact",
        "apikey": ALPHA_KEY,
        "datatype": "json"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js.keys() if "Time Series" in k]
    if not key:
        # Alpha Vantage replies HTTP 200 with a prose blob - not an error status - when
        # the free tier's daily cap is hit or the endpoint has moved to premium, so
        # raise_for_status() above sees nothing wrong. Say so and fall back.
        print("Alpha Vantage returned no series:", str(list(js.values())[0])[:150])
        use_alpha = False

if use_alpha:
    series = js[key[0]]
    df_api = (pd.DataFrame(series).T
              .rename_axis('date')
              .reset_index())
    # keep a couple columns and coerce types
    df_api = df_api[['date', '4. close']].rename(columns={'4. close': 'close'})
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

if not use_alpha:
    import yfinance as yf
    df_api = yf.download(SYMBOL, period="6mo", interval="1d", auto_adjust=False,
                          multi_level_index=False).reset_index()[['Date','Close']]
    df_api.columns = ['date','close']

df_api = df_api.sort_values('date').reset_index(drop=True)
msgs = validate_df(df_api, required_cols=['date','close'], dtypes_map={'date':'datetime64[ns]','close':'float'})
print(msgs)

# Additional validation
print("Shape:", df_api.shape)
print("Missing values:")
print(df_api.isna().sum())

assert df_api.shape[0] > 0, "API dataset has no rows."
assert {'date', 'close'}.issubset(df_api.columns), "Required columns are missing."
assert pd.api.types.is_datetime64_any_dtype(df_api['date']), "date must be datetime."
assert pd.api.types.is_numeric_dtype(df_api['close']), "close must be numeric."
assert (df_api['close'] > 0).all(), "Close prices must be positive."

print("API validation passed.")

fname = safe_filename(prefix="api", meta={"source": "alpha" if use_alpha else "yfinance", "symbol": SYMBOL})
out_path = DATA_RAW / fname
df_api.to_csv(out_path, index=False)
print("Saved:", out_path)

Using Alpha Vantage: True
{'na_total': 'Total NA values: 0'}
Shape: (100, 2)
Missing values:
date     0
close    0
dtype: int64
API validation passed.
Saved: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/homework/homework4/data/raw/api_source-alpha_symbol-AAPL_20260819-235247.csv


## Scraping a Simple Public Table with BeautifulSoup
*Use responsibly. Provide a polite User-Agent and delays if looping.*

In [9]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_Dow_Jones_Industrial_Average_companies"

headers = {
    "User-Agent": "Mozilla/5.0 educational-data-ingestion-project"
}

try:
    # 1. Request the public webpage
    response = requests.get(
        SCRAPE_URL,
        headers=headers,
        timeout=30
    )
    response.raise_for_status()

    # 2. Parse HTML with BeautifulSoup
    soup = BeautifulSoup(response.text, "html.parser")

    # Preferred selector
    table = soup.select_one("table.wikitable")

    # Fallback: search for a table containing expected headers
    if table is None:
        for candidate in soup.find_all("table"):
            header_text = candidate.get_text(" ", strip=True)
            if "Company" in header_text and "Symbol" in header_text:
                table = candidate
                break

    if table is None:
        raise ValueError("Could not find the Dow Jones components table.")

    # 3. Extract column names
    header_cells = table.find("tr").find_all(["th", "td"])
    column_names = [
        cell.get_text(" ", strip=True)
        for cell in header_cells
    ]

    # 4. Extract rows
    records = []

    for row in table.find_all("tr")[1:]:
        cells = row.find_all(["th", "td"])

        values = [
            cell.get_text(" ", strip=True)
            for cell in cells
        ]

        if len(values) == len(column_names):
            records.append(values)

    # 5. Build DataFrame
    df_scrape = pd.DataFrame(
        records,
        columns=column_names
    )

except Exception as e:
    print("Scraping failed:", e)
    raise

print("Scraped shape:", df_scrape.shape)
print("Columns:", df_scrape.columns.tolist())

df_scrape.head()

Scraped shape: (30, 6)
Columns: ['Company', 'Exchange', 'Symbol', 'Sector', 'Date added', 'Notes']


,Company,Exchange,Symbol,Sector,Date added,Notes
0,3M,NYSE,MMM,Industrials,1976-08-09,As Minnesota Mining and Manufacturing
1,Alphabet (Class A),NASDAQ,GOOGL,Communication Services,2026-06-29,
2,American Express,NYSE,AXP,Financials,1982-08-30,
3,Amgen,NASDAQ,AMGN,Health Care,2020-08-31,
4,Amazon,NASDAQ,AMZN,Consumer Discretionary,2024-02-26,


In [10]:
# Parse date type
df_scrape["Date added"] = pd.to_datetime(
    df_scrape["Date added"],
    errors="coerce"
)

# Required columns
required_scrape_cols = [
    "Company",
    "Exchange",
    "Symbol",
    "Sector",
    "Date added"
]

missing_cols = [
    col for col in required_scrape_cols
    if col not in df_scrape.columns
]

assert len(missing_cols) == 0, \
    f"Missing required columns: {missing_cols}"

# Shape validation
print("Shape:", df_scrape.shape)
assert df_scrape.shape[0] > 0, "Scraped dataset has no rows."

# Missing-value counts
print("\nMissing values:")
print(df_scrape.isna().sum())

# Text validation
for col in ["Company", "Exchange", "Symbol", "Sector"]:
    assert df_scrape[col].notna().all(), \
        f"{col} contains missing values."

    assert df_scrape[col].str.strip().ne("").all(), \
        f"{col} contains empty strings."

# Date validation
assert pd.api.types.is_datetime64_any_dtype(
    df_scrape["Date added"]
), "Date added must be datetime."

# Numeric validation using the year from Date added
year_added = df_scrape["Date added"].dt.year

assert pd.api.types.is_numeric_dtype(
    year_added
), "Year added must be numeric."

# Basic rule: ticker symbols should be unique
assert df_scrape["Symbol"].duplicated().sum() == 0, \
    "Duplicate ticker symbols found."

print("\nScraping validation passed.")

Shape: (30, 6)

Missing values:
Company       0
Exchange      0
Symbol        0
Sector        0
Date added    0
Notes         0
dtype: int64

Scraping validation passed.


In [11]:
fname_scrape = safe_filename(
    prefix="scrape",
    meta={
        "site": "wikipedia",
        "table": "djia"
    }
)

scrape_path = DATA_RAW / fname_scrape

df_scrape.to_csv(
    scrape_path,
    index=False
)

print("Saved:", scrape_path)

Saved: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong/homework/homework4/data/raw/scrape_site-wikipedia_table-djia_20260819-235248.csv


## Data Sources, Parameters, and Validation Logic

### API Source
- Source: Alpha Vantage
- Endpoint: https://www.alphavantage.co/query
- Ticker: AAPL
- Function: TIME_SERIES_DAILY
- Output size: compact
- Data format: JSON
- API key: loaded securely from `.env`

The API response is converted to a pandas DataFrame. The `date` column is parsed as datetime and the `close` column is parsed as numeric.

API validation checks:
- Required columns: `date` and `close`
- Dataset shape is non-empty
- Missing-value counts are reported
- `date` is a datetime type
- `close` is numeric
- Closing prices must be positive

### Scraping Source
- Source: Wikipedia
- URL: https://en.wikipedia.org/wiki/List_of_Dow_Jones_Industrial_Average_companies
- Table: Dow Jones Industrial Average component companies

The page is requested with `requests` and parsed with `BeautifulSoup`. The HTML table is converted into a pandas DataFrame.

Scraping validation checks:
- Required columns are present
- Dataset shape is non-empty
- Missing-value counts are reported
- Text fields are non-empty
- `Date added` is parsed as datetime
- Year values are validated as numeric
- Ticker symbols must be unique

In [12]:
import subprocess

result = subprocess.run(
    ["git", "ls-files", ".env"],
    cwd=ROOT,
    capture_output=True,
    text=True
)

tracked_env = result.stdout.strip()

print(".env tracked by Git:", bool(tracked_env))

assert tracked_env == "", \
    ".env should not be committed to Git."

print(".env is not tracked by Git.")

.env tracked by Git: False
.env is not tracked by Git.


In [13]:
from pathlib import Path

HOMEWORK_DIR = Path.cwd()
REPO_ROOT = HOMEWORK_DIR.parents[1]

gitignore_path = REPO_ROOT / ".gitignore"

gitignore_text = gitignore_path.read_text()

assert ".env" in gitignore_text, \
    ".env is not listed in .gitignore."

print("Repository root:", REPO_ROOT)
print(".env is correctly listed in .gitignore.")

Repository root: /Users/juan/Desktop/nyu bootcamp/bootcamp_henry_dong
.env is correctly listed in .gitignore.


## Assumptions & Risks

### Assumptions
- AAPL is used as a representative market-related ticker.
- Alpha Vantage daily price data is sufficient for this ingestion exercise.
- The Dow Jones component table on Wikipedia is publicly accessible and maintains a similar HTML structure.
- The current list contains 30 Dow Jones component companies.

### Risks
- The Alpha Vantage API may be temporarily unavailable or subject to rate limits.
- API response structure may change in the future.
- The Wikipedia page or HTML table structure may change and break the scraper.
- Dow Jones membership can change over time, so future runs may produce different records.

## Summary

This notebook implements a reproducible data ingestion workflow using two sources:

1. AAPL daily market data retrieved from Alpha Vantage using an API key stored in `.env`.
2. Dow Jones Industrial Average component data scraped from Wikipedia using `requests` and `BeautifulSoup`.

Both datasets are converted to pandas DataFrames, parsed into appropriate data types, validated, and saved as timestamped CSV files in `data/raw/`.